In [1]:
import pandas as pd

df_equipos = pd.read_parquet("equipos.parquet")
df_partidos = pd.read_parquet("partidos.parquet")
df_clasificacion = pd.read_parquet("clasificacion.parquet")

df_equipos.head()

,equipo_id,nombre_equipo,codigo_equipo,pais,anio_fundacion,es_seleccion_nacional,logo_url,competencia_id,temporada,fecha_extraccion,extraido_por,endpoint_origen
0,1,Belgium,BEL,Belgium,1895,True,https://media.api-sports.io/football/teams/1.png,1,2022,2026-08-17T17:19:57,Pablo y Pedro,/teams
1,2,France,FRA,France,1919,True,https://media.api-sports.io/football/teams/2.png,1,2022,2026-08-17T17:19:57,Pablo y Pedro,/teams
2,3,Croatia,CRO,Croatia,1912,True,https://media.api-sports.io/football/teams/3.png,1,2022,2026-08-17T17:19:57,Pablo y Pedro,/teams
3,6,Brazil,BRA,Brazil,1914,True,https://media.api-sports.io/football/teams/6.png,1,2022,2026-08-17T17:19:57,Pablo y Pedro,/teams
4,7,Uruguay,URU,Uruguay,1900,True,https://media.api-sports.io/football/teams/7.png,1,2022,2026-08-17T17:19:57,Pablo y Pedro,/teams


In [2]:
df_partidos.head()

,partido_id,competencia_id,competencia_nombre,temporada,ronda,fecha_partido,zona_horaria,estado_partido,minuto_transcurrido,arbitro,...,equipo_visitante_nombre,gano_local,gano_visitante,goles_local,goles_visitante,penales_local,penales_visitante,fecha_extraccion,extraido_por,endpoint_origen
0,855736,1,World Cup,2022,Group Stage - 1,2022-11-20 16:00:00+00:00,UTC,Match Finished,90,D. Orsato,...,Ecuador,False,True,0,2,<NA>,<NA>,2026-08-17T17:19:57,Pablo y Pedro,/fixtures
1,855735,1,World Cup,2022,Group Stage - 1,2022-11-21 13:00:00+00:00,UTC,Match Finished,90,Raphael Claus,...,Iran,True,False,6,2,<NA>,<NA>,2026-08-17T17:19:57,Pablo y Pedro,/fixtures
2,855734,1,World Cup,2022,Group Stage - 1,2022-11-21 16:00:00+00:00,UTC,Match Finished,90,Wilton Pereira Sampaio,...,Netherlands,False,True,0,2,<NA>,<NA>,2026-08-17T17:19:57,Pablo y Pedro,/fixtures
3,866681,1,World Cup,2022,Group Stage - 1,2022-11-21 19:00:00+00:00,UTC,Match Finished,90,Abdulrahman Al Jassim,...,Wales,<NA>,<NA>,1,1,<NA>,<NA>,2026-08-17T17:19:57,Pablo y Pedro,/fixtures
4,855737,1,World Cup,2022,Group Stage - 1,2022-11-22 10:00:00+00:00,UTC,Match Finished,90,S. Vinčić,...,Saudi Arabia,False,True,1,2,<NA>,<NA>,2026-08-17T17:19:57,Pablo y Pedro,/fixtures


In [3]:
df_clasificacion.head()

,grupo,posicion,equipo_id,nombre_equipo,puntos,partidos_jugados,partidos_ganados,partidos_empatados,partidos_perdidos,goles_favor,...,diferencia_gol,forma_reciente,estado_clasificacion,descripcion_clasificacion,fecha_actualizacion,competencia_id,temporada,fecha_extraccion,extraido_por,endpoint_origen
0,Group A,1,1118,Netherlands,7,3,2,1,0,5,...,4,WDW,same,Promotion - World Cup (Play Offs),2022-12-12 00:00:00+00:00,1,2022,2026-08-17T17:19:57,Pablo y Pedro,/standings
1,Group A,2,13,Senegal,6,3,2,0,1,5,...,1,WWL,same,Promotion - World Cup (Play Offs),2022-12-12 00:00:00+00:00,1,2022,2026-08-17T17:19:57,Pablo y Pedro,/standings
2,Group A,3,2382,Ecuador,4,3,1,1,1,4,...,1,LDW,same,NaN,2022-12-12 00:00:00+00:00,1,2022,2026-08-17T17:19:57,Pablo y Pedro,/standings
3,Group A,4,1569,Qatar,0,3,0,0,3,1,...,-6,LLL,same,NaN,2022-12-12 00:00:00+00:00,1,2022,2026-08-17T17:19:57,Pablo y Pedro,/standings
4,Group B,1,10,England,7,3,2,1,0,9,...,7,WDW,same,Promotion - World Cup (Play Offs),2022-12-12 00:00:00+00:00,1,2022,2026-08-17T17:19:57,Pablo y Pedro,/standings


## 1. ¿Cuántos equipos participaron en la competición?

In [4]:
num_equipos = df_equipos["equipo_id"].nunique()
print(f"Número de equipos: {num_equipos}")

Número de equipos: 32


## 2. ¿Cuántos partidos se jugaron en cada ronda?

In [5]:
partidos_por_ronda = df_partidos.groupby("ronda").size().reset_index(name="cantidad_partidos")
partidos_por_ronda = partidos_por_ronda.sort_values("cantidad_partidos", ascending=False).reset_index(drop=True)
partidos_por_ronda

,ronda,cantidad_partidos
0,Group Stage - 2,16
1,Group Stage - 1,16
2,Group Stage - 3,16
3,Round of 16,8
4,Quarter-finals,4
5,Semi-finals,2
6,3rd Place Final,1
7,Final,1


## 3. ¿Cuál fue el partido con mayor cantidad total de goles?

In [6]:
df_partidos["total_goles"] = df_partidos["goles_local"] + df_partidos["goles_visitante"]
max_goles = df_partidos["total_goles"].max()

df_partidos.query("total_goles == @max_goles")[
    ["partido_id", "equipo_local_nombre", "equipo_visitante_nombre", "goles_local", "goles_visitante", "total_goles"]
]

,partido_id,equipo_local_nombre,equipo_visitante_nombre,goles_local,goles_visitante,total_goles
1,855735,England,Iran,6,2,8


## 4. ¿Cuál fue el equipo que anotó más goles durante toda la competición?

In [7]:
goles_local = df_partidos.groupby("equipo_local_nombre")["goles_local"].sum()
goles_visitante = df_partidos.groupby("equipo_visitante_nombre")["goles_visitante"].sum()

goles_totales = goles_local.add(goles_visitante, fill_value=0).reset_index()
goles_totales.columns = ["equipo", "goles_totales"]

max_goles_equipo = goles_totales["goles_totales"].max()
goles_totales.query("goles_totales == @max_goles_equipo")

,equipo,goles_totales
11,France,16


## 5. ¿Cuál fue el equipo que ganó más partidos?

In [8]:
victorias_local = df_partidos[df_partidos["gano_local"] == True].groupby("equipo_local_nombre").size()
victorias_visitante = df_partidos[df_partidos["gano_visitante"] == True].groupby("equipo_visitante_nombre").size()

victorias = victorias_local.add(victorias_visitante, fill_value=0).astype(int).reset_index()
victorias.columns = ["equipo", "victorias"]

max_victorias = victorias["victorias"].max()
victorias.query("victorias == @max_victorias")

,equipo,victorias
0,Argentina,6


## 6. ¿Cuál fue el equipo con la mejor diferencia de gol dentro de cada grupo?

In [9]:
mejor_diferencia_grupo = df_clasificacion.groupby("grupo")["diferencia_gol"].transform("max")
mejores = df_clasificacion[df_clasificacion["diferencia_gol"] == mejor_diferencia_grupo]

mejores[["grupo", "nombre_equipo", "diferencia_gol"]].sort_values(["grupo", "nombre_equipo"]).reset_index(drop=True)

,grupo,nombre_equipo,diferencia_gol
0,Group A,Netherlands,4
1,Group B,England,7
2,Group C,Argentina,3
3,Group D,France,3
4,Group E,Spain,6
5,Group F,Croatia,3
6,Group F,Morocco,3
7,Group G,Brazil,2
8,Group H,Portugal,2


## 7. ¿Qué equipos terminaron en la primera posición de cada grupo?

In [10]:
df_clasificacion.query("posicion == 1")[["grupo", "nombre_equipo", "puntos"]].sort_values("grupo").reset_index(drop=True)

,grupo,nombre_equipo,puntos
0,Group A,Netherlands,7
1,Group B,England,7
2,Group C,Argentina,6
3,Group D,France,6
4,Group E,Japan,6
5,Group F,Morocco,7
6,Group G,Brazil,6
7,Group H,Portugal,6


## 8. ¿Cuál fue el estadio en el que se disputaron más partidos?

In [11]:
partidos_por_estadio = df_partidos.groupby("estadio_nombre").size().reset_index(name="cantidad_partidos")
max_partidos_estadio = partidos_por_estadio["cantidad_partidos"].max()

partidos_por_estadio.query("cantidad_partidos == @max_partidos_estadio")

,estadio_nombre,cantidad_partidos
6,Lusail Iconic Stadium,10


## 9. ¿Existen partidos duplicados según el campo partido_id?

In [12]:
duplicados = df_partidos["partido_id"].duplicated().sum()
print(f"Cantidad de partido_id duplicados: {duplicados}")

Cantidad de partido_id duplicados: 0


## 10. ¿Cuántos valores nulos tiene cada columna de cada archivo?

In [13]:
print("Valores nulos en equipos.parquet:")
print(df_equipos.isnull().sum())

Valores nulos en equipos.parquet:
equipo_id                0
nombre_equipo            0
codigo_equipo            0
pais                     0
anio_fundacion           0
es_seleccion_nacional    0
logo_url                 0
competencia_id           0
temporada                0
fecha_extraccion         0
extraido_por             0
endpoint_origen          0
dtype: int64


In [14]:
print("Valores nulos en partidos.parquet:")
print(df_partidos.isnull().sum())

Valores nulos en partidos.parquet:
partido_id                  0
competencia_id              0
competencia_nombre          0
temporada                   0
ronda                       0
fecha_partido               0
zona_horaria                0
estado_partido              0
minuto_transcurrido         0
arbitro                     0
estadio_id                 56
estadio_nombre              0
estadio_ciudad              0
equipo_local_id             0
equipo_local_nombre         0
equipo_visitante_id         0
equipo_visitante_nombre     0
gano_local                 10
gano_visitante             10
goles_local                 0
goles_visitante             0
penales_local              59
penales_visitante          59
fecha_extraccion            0
extraido_por                0
endpoint_origen             0
total_goles                 0
dtype: int64


In [15]:
print("Valores nulos en clasificacion.parquet:")
print(df_clasificacion.isnull().sum())

Valores nulos en clasificacion.parquet:
grupo                         0
posicion                      0
equipo_id                     0
nombre_equipo                 0
puntos                        0
partidos_jugados              0
partidos_ganados              0
partidos_empatados            0
partidos_perdidos             0
goles_favor                   0
goles_contra                  0
diferencia_gol                0
forma_reciente                0
estado_clasificacion          0
descripcion_clasificacion    16
fecha_actualizacion           0
competencia_id                0
temporada                     0
fecha_extraccion              0
extraido_por                  0
endpoint_origen               0
dtype: int64
